In [1]:
import pandas as pd
df=pd.read_csv("Support_Ticket.csv")

In [21]:
df.head()

,subject,body,type,queue,priority,language
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Incident,Technical Support,high,de
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Incident,Technical Support,high,en
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Request,Returns and Exchanges,medium,en
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Request,Billing and Payments,low,en
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Problem,Sales and Pre-Sales,medium,en


In [3]:
df.shape

(28587, 6)

In [4]:
df.isnull().sum()

subject     3838
body           0
type           0
queue          0
priority       0
language       0
dtype: int64

In [5]:
df = df.dropna(subset=['subject'])

In [6]:
df.isnull().sum()

subject     0
body        0
type        0
queue       0
priority    0
language    0
dtype: int64

In [7]:
df.shape

(24749, 6)

In [16]:
df['type'].value_counts()

type
Incident    9951
Request     7081
Problem     5233
Change      2484
Name: count, dtype: int64

In [17]:
df['queue'].value_counts()

queue
Technical Support                  7260
Product Support                    4563
Customer Service                   3733
IT Support                         2943
Billing and Payments               2427
Returns and Exchanges              1226
Service Outages and Maintenance    1024
Sales and Pre-Sales                 763
Human Resources                     473
General Inquiry                     337
Name: count, dtype: int64

In [18]:
df['priority'].value_counts()

priority
medium    9978
high      9655
low       5116
Name: count, dtype: int64

In [19]:
df['language'].value_counts()

language
en    13731
de    11018
Name: count, dtype: int64

In [22]:
df = df[df['language'] == 'en'].copy()

In [24]:
df.shape

(13731, 6)

In [26]:
df['type'].value_counts()

type
Incident    5531
Request     3929
Problem     2874
Change      1397
Name: count, dtype: int64

In [27]:
df['queue'].value_counts()

queue
Technical Support                  3971
Product Support                    2608
Customer Service                   2045
IT Support                         1609
Billing and Payments               1349
Returns and Exchanges               686
Service Outages and Maintenance     581
Sales and Pre-Sales                 410
Human Resources                     285
General Inquiry                     187
Name: count, dtype: int64

In [28]:
df['priority'].value_counts()

priority
medium    5595
high      5304
low       2832
Name: count, dtype: int64

In [31]:
df.drop(columns=["language"], inplace=True)

In [34]:
df.head()

,subject,body,type,queue,priority
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Incident,Technical Support,high
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Request,Returns and Exchanges,medium
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Request,Billing and Payments,low
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Problem,Sales and Pre-Sales,medium
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Request,Technical Support,high


In [35]:
import re

def clean_text(text):
    text = str(text).lower()

    # Remove newline and tab characters
    text = re.sub(r'[\n\r\t]+', ' ', text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [36]:
X = df[['subject', 'body', 'type']].copy()

X['subject'] = X['subject'].apply(clean_text)
X['body'] = X['body'].apply(clean_text)

In [37]:
X['text'] = X['subject'] + ' ' + X['body']

In [46]:
X.head()

,subject,body,type,text
1,account disruption,dear customer support team n ni am writing to ...,Incident,account disruption dear customer support team ...
2,query about smart home system integration feat...,dear customer support team n ni hope this mess...,Request,query about smart home system integration feat...
3,inquiry regarding invoice details,dear customer support team n ni hope this mess...,Request,inquiry regarding invoice details dear custome...
4,question about marketing agency software compa...,dear support team n ni hope this message reach...,Problem,question about marketing agency software compa...
5,feature query,dear customer support n ni hope this message r...,Request,feature query dear customer support n ni hope ...


In [ ]:
stratify_label = (
    df['queue'].astype(str) + '_' +
    df['priority'].astype(str)
)

In [41]:
y_queue=df['queue']
y_priority=df['priority']

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_queue_train, y_queue_test, y_priority_train, y_priority_test = train_test_split(
    X,
    y_queue,
    y_priority,
    test_size=0.2,
    random_state=42,
    stratify=stratify_label
)

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)


X_train_text = tfidf.fit_transform(X_train['text'])
X_test_text = tfidf.transform(X_test['text'])

In [59]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True
)

X_train_type = encoder.fit_transform(X_train[['type']])
X_test_type = encoder.transform(X_test[['type']])

In [63]:
X_train.shape

(10984, 4)

In [64]:
X_test.shape

(2747, 4)

In [65]:
from scipy.sparse import hstack

X_train_final = hstack([
    X_train_text,
    X_train_type
])

X_test_final = hstack([
    X_test_text,
    X_test_type
])

In [66]:
from sklearn.linear_model import LogisticRegression

queue_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    class_weight='balanced'
)

priority_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    class_weight='balanced'
)

In [67]:
queue_model.fit(X_train_final, y_queue_train)

priority_model.fit(X_train_final, y_priority_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

In [68]:
from sklearn.svm import LinearSVC

queue_svm = LinearSVC(
    C=1.0,
    class_weight='balanced'
)

priority_svm = LinearSVC(
    C=1.0,
    class_weight='balanced'
)

In [69]:
queue_svm.fit(X_train_final, y_queue_train)

priority_svm.fit(X_train_final, y_priority_train)

,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [70]:
from sklearn.naive_bayes import MultinomialNB

queue_nb = MultinomialNB()
priority_nb = MultinomialNB()

In [71]:
queue_nb.fit(X_train_final, y_queue_train)

priority_nb.fit(X_train_final, y_priority_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](3,)","[4.24e+03,2.26e+03,4.48e+03]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](3,)","[-0.951,-1.58,-0.898]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U6](3,)","['high','low','medium']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](3, 15004)","[[1.57,1.58,3.92,1.36,0.981,1.06,0,0.353,3.24,0.384,0.633,0.408,0.207,2.82,2.3,1.22,0.592,2.49,2.29,0.551,0.201,0.278,0.224,0.224,2.15,0.83,1.55,2.56,0.587,8.39,...,0.586,0.63,0.276,0.966,11,1.72,2.06,1.1,0.68,6.36,0.817,0.121,0.876,10.9,2.63,0.805,1.1,2.46,0.596,0,0.548,0,4,2.49,1.93,1.28,452,1.86e+03,857,1.07e+03], [0.478,0.381,0.947,0.135,0,0.67,0.745,0.215,0.603,0.152,0,0.145,0,0.144,0,0.312,0,0.583,2.08,0.826,0,1.15,0.181,0.181,0.877,0.508,0.17,1.43,0.682,3.27,...,0.275,0.128,0.142,0.241,6.75,1.74,2.09,0.566,0.407,3.58,0.367,0.223,1.59,11.5,1.61,0,0.283,1.45,0.655,0.135,0,0,2.68,0.909,1.96,0,234,822,468,740], [1.51,1.52,2.36,0,0.877,0.823,0.158,1.2,5.09,1.46,0.608,0.599,1.26,1.73,1.24,1.57,0.717,1.44,6.38,1.09,0.953,0.193,0.561,0.561,2.4,1.08,1.31,1.76,0,6.07,...,0.85,0.966,0.848,0.507,13.1,2.01,1.9,1.23,0.433,5.82,0.769,0.878,2.07,15.4,3.61,0.409,0.615,5.41,0.595,0.894,0.62,2.01,3.9,1.81,2.28,2.53,450,1.69e+03,987,1.35e+03]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](3, 15004)","[[-9.97,-9.97,-9.33,-10.1,-10.2,-10.2,-10.9,-10.6,-9.47,-10.6,-10.4,-10.6,-10.7,-9.58,-9.72,-10.1,-10.5,-9.67,-9.73,-10.5,-10.7,-10.7,-10.7,-10.7,-9.77,-10.3,-9.98,-9.65,-10.5,-8.68,...,-10.5,-10.4,-10.7,-10.2,-8.44,-9.92,-9.8,-10.2,-10.4,-8.92,-10.3,-10.8,-10.3,-8.44,-9.63,-10.3,-10.2,-9.67,-10.4,-10.9,-10.5,-10.9,-9.31,-9.67,-9.84,-10.1,-4.8,-3.39,-4.16,-3.94], [-10.1,-10.2,-9.83,-10.4,-10.5,-9.99,-9.94,-10.3,-10,-10.4,-10.5,-10.4,-10.5,-10.4,-10.5,-10.2,-10.5,-10,-9.38,-9.9,-10.5,-9.74,-10.3,-10.3,-9.87,-10.1,-10.3,-9.61,-9.98,-9.05,...,-10.3,-10.4,-10.4,-10.3,-8.45,-9.49,-9.37,-10.1,-10.2,-8.98,-10.2,-10.3,-9.55,-7.98,-9.54,-10.5,-10.3,-9.6,-10,-10.4,-10.5,-10.5,-9.2,-9.85,-9.41,-10.5,-5.04,-3.79,-4.35,-3.89], [-10,-10,-9.74,-11,-10.3,-10.4,-10.8,-10.2,-9.15,-10.1,-10.5,-10.5,-10.1,-9.95,-10.1,-10,-10.4,-10.1,-8.95,-10.2,-10.3,-10.8,-10.5,-10.5,-9.73,-10.2,-10.1,-9.94,-11,-9,...,-10.3,-10.3,-10.3,-10.5,-8.31,-9.85,-9.89,-10.2,-10.6,-9.03,-10.4,-10.3,-9.83,-8.15,-9.42,-10.6,-10.5,-9.09,-10.5,-10.3,-10.5,-9.85,-9.36,-9.92,-9.77,-9.69,-4.84,-3.52,-4.06,-3.74]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,15004


In [ ]:


queue_lr_pred = queue_model.predict(X_test_final)
priority_lr_pred = priority_model.predict(X_test_final)


queue_svm_pred = queue_svm.predict(X_test_final)
priority_svm_pred = priority_svm.predict(X_test_final)



queue_nb_pred = queue_nb.predict(X_test_final)
priority_nb_pred = priority_nb.predict(X_test_final)

In [82]:
from sklearn.metrics import classification_report

# =========================
# QUEUE - ALL MODELS
# =========================

print("\n" + "="*80)
print("QUEUE - LOGISTIC REGRESSION")
print("="*80)
print(classification_report(
    y_queue_test,
    queue_lr_pred,
    digits=4,
    zero_division=0
))

print("\n" + "="*80)
print("QUEUE - LINEAR SVM")
print("="*80)
print(classification_report(
    y_queue_test,
    queue_svm_pred,
    digits=4,
    zero_division=0
))

print("\n" + "="*80)
print("QUEUE - NAIVE BAYES")
print("="*80)
print(classification_report(
    y_queue_test,
    queue_nb_pred,
    digits=4,
    zero_division=0
))


# =========================
# PRIORITY - ALL MODELS
# =========================

print("\n" + "="*80)
print("PRIORITY - LOGISTIC REGRESSION")
print("="*80)
print(classification_report(
    y_priority_test,
    priority_lr_pred,
    digits=4,
    zero_division=0
))

print("\n" + "="*80)
print("PRIORITY - LINEAR SVM")
print("="*80)
print(classification_report(
    y_priority_test,
    priority_lr_pred,
    digits=4,
    zero_division=0
))

print("\n" + "="*80)
print("PRIORITY - NAIVE BAYES")
print("="*80)
print(classification_report(
    y_priority_test,
    priority_nb_pred,
    digits=4,
    zero_division=0
))


QUEUE - LOGISTIC REGRESSION
                                 precision    recall  f1-score   support

           Billing and Payments     0.7429    0.7704    0.7564       270
               Customer Service     0.4108    0.4279    0.4192       409
                General Inquiry     0.3559    0.5676    0.4375        37
                Human Resources     0.3523    0.5439    0.4276        57
                     IT Support     0.4195    0.4938    0.4536       322
                Product Support     0.5039    0.3716    0.4278       522
          Returns and Exchanges     0.4059    0.5036    0.4495       137
            Sales and Pre-Sales     0.2910    0.6707    0.4059        82
Service Outages and Maintenance     0.5669    0.7672    0.6520       116
              Technical Support     0.5749    0.4440    0.5011       795

                       accuracy                         0.4929      2747
                      macro avg     0.4624    0.5561    0.4931      2747
                   w

In [84]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

param_grid = {
    'C': [0.01, 0.1, 0.5, 1, 2, 5, 10]
}

queue_grid = GridSearchCV(
    LinearSVC(class_weight='balanced'),
    param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

queue_grid.fit(X_train_final, y_queue_train)

print(queue_grid.best_params_)
print(queue_grid.best_score_)

{'C': 5}
0.6063792189910053


In [85]:
priority_grid = GridSearchCV(
    LinearSVC(class_weight='balanced'),
    param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

priority_grid.fit(X_train_final, y_priority_train)

print(priority_grid.best_params_)
print(priority_grid.best_score_)

{'C': 5}
0.6201520296159387


In [86]:
from sklearn.svm import LinearSVC

queue_svm_best = LinearSVC(
    C=5,
    class_weight='balanced'
)

priority_svm_best = LinearSVC(
    C=5,
    class_weight='balanced'
)

queue_svm_best.fit(X_train_final, y_queue_train)

priority_svm_best.fit(X_train_final, y_priority_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",5
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo 

In [87]:
queue_svm_best_pred = queue_svm_best.predict(X_test_final)

priority_svm_best_pred = priority_svm_best.predict(X_test_final)

In [88]:
from sklearn.metrics import classification_report

print("QUEUE")
print(classification_report(
    y_queue_test,
    queue_svm_best_pred,
    digits=4,
    zero_division=0
))

print("\nPRIORITY")
print(classification_report(
    y_priority_test,
    priority_svm_best_pred,
    digits=4,
    zero_division=0
))

QUEUE
                                 precision    recall  f1-score   support

           Billing and Payments     0.8208    0.8481    0.8342       270
               Customer Service     0.5489    0.5623    0.5556       409
                General Inquiry     0.7308    0.5135    0.6032        37
                Human Resources     0.7714    0.4737    0.5870        57
                     IT Support     0.5601    0.5932    0.5762       322
                Product Support     0.5833    0.5766    0.5800       522
          Returns and Exchanges     0.6423    0.5766    0.6077       137
            Sales and Pre-Sales     0.5506    0.5976    0.5731        82
Service Outages and Maintenance     0.7339    0.7845    0.7583       116
              Technical Support     0.6491    0.6491    0.6491       795

                       accuracy                         0.6305      2747
                      macro avg     0.6591    0.6175    0.6324      2747
                   weighted avg     0.6321 

In [89]:
from sklearn.metrics import accuracy_score, f1_score

print("QUEUE")
print("Accuracy:", accuracy_score(y_queue_test, queue_svm_best_pred))
print("Macro F1:", f1_score(y_queue_test, queue_svm_best_pred, average='macro'))
print("Weighted F1:", f1_score(y_queue_test, queue_svm_best_pred, average='weighted'))

print("\nPRIORITY")
print("Accuracy:", accuracy_score(y_priority_test, priority_svm_best_pred))
print("Macro F1:", f1_score(y_priority_test, priority_svm_best_pred, average='macro'))
print("Weighted F1:", f1_score(y_priority_test, priority_svm_best_pred, average='weighted'))

QUEUE
Accuracy: 0.6305060065526028
Macro F1: 0.6324242814083546
Weighted F1: 0.6300411541142961

PRIORITY
Accuracy: 0.6497997815799054
Macro F1: 0.6386695175580768
Weighted F1: 0.6491732625581217
